# Instacart Lakehouse Analytics — Silver Cleaning and Data Model

## Objective

This notebook transforms the raw Bronze tables into cleaned, standardized, and enriched Silver tables.

The Silver layer combines transactional data, validates key fields, enriches products with aisle and department attributes, and creates an analysis-ready order-item dataset.

## Inputs

- `bronze_orders`
- `bronze_products`
- `bronze_aisles`
- `bronze_departments`
- `bronze_order_products_prior`
- `bronze_order_products_train`

## Outputs

- `silver_orders`
- `silver_product_dimension`
- `silver_order_items`

## 1. Load Bronze Tables

Load the managed Bronze Delta tables from Unity Catalog as Spark DataFrames for cleaning and transformation.

In [0]:
from pyspark.sql import functions as F

orders = spark.table("workspace.default.bronze_orders")
products = spark.table("workspace.default.bronze_products")
aisles = spark.table("workspace.default.bronze_aisles")
departments = spark.table("workspace.default.bronze_departments")
prior = spark.table("workspace.default.bronze_order_products_prior")
train = spark.table("workspace.default.bronze_order_products_train")

## 2. Combine Transaction Sources

The prior and train order-product datasets have the same structure but represent different evaluation groups.

They are standardized with an `eval_set` label and combined into one transaction-level DataFrame for downstream processing.

In [0]:
# Add source labels before combining the two transaction datasets
prior_clean = prior.withColumn("eval_set", F.lit("prior"))
train_clean = train.withColumn("eval_set", F.lit("train"))

order_products = prior_clean.unionByName(train_clean)

## 3. Build the Product Dimension

Join products with aisle and department lookup tables to create a business-friendly product dimension.

This dimension provides descriptive context for each product and supports downstream product, aisle, and department analysis.

In [0]:
# Enrich products with aisle and department names
silver_product_dimension = (
    products
    .join(aisles, on="aisle_id", how="left")
    .join(departments, on="department_id", how="left")
    .select(
        "product_id",
        "product_name",
        "aisle_id",
        "aisle",
        "department_id",
        "department"
    )
    .dropDuplicates(["product_id"])
)

## 4. Clean the Orders Dataset

Select the required order fields, remove duplicate order records, and exclude rows with missing order or customer identifiers.

The resulting table represents one clean record per order.

In [0]:
silver_orders = (
    orders
    .select(
        "order_id",
        "user_id",
        "eval_set",
        "order_number",
        "order_dow",
        "order_hour_of_day",
        "days_since_prior_order"
    )
    .dropDuplicates(["order_id"])
    .filter(F.col("order_id").isNotNull())
    .filter(F.col("user_id").isNotNull())
)

## 5. Create the Enriched Order-Item Table

Join the combined order-product transactions with cleaned orders and the product dimension.

The resulting table contains customer, product, department, basket-position, reorder, and order-timing attributes at the order-item level.

In [0]:
# Create the central transaction-level Silver table
silver_order_items = (
    order_products
    .join(
        silver_orders.select(
            "order_id",
            "user_id",
            "order_number",
            "order_dow",
            "order_hour_of_day",
            "days_since_prior_order"
        ),
        on="order_id",
        how="inner"
    )
    .join(
        silver_product_dimension,
        on="product_id",
        how="left"
    )
    .select(
        "order_id",
        "user_id",
        "product_id",
        "product_name",
        "aisle_id",
        "aisle",
        "department_id",
        "department",
        "add_to_cart_order",
        "reordered",
        "order_number",
        "order_dow",
        "order_hour_of_day",
        "days_since_prior_order",
        "eval_set"
    )
)

## 6. Validate Silver Outputs

Review row counts and sample records to confirm that the joins and transformations produced complete, analysis-ready datasets.

In [0]:
print("silver_orders:", silver_orders.count())
print("silver_product_dimension:", silver_product_dimension.count())
print("silver_order_items:", silver_order_items.count())

silver_orders: 109467
silver_product_dimension: 49688
silver_order_items: 1130617


## 7. Write Silver Delta Tables

Store the cleaned and enriched datasets as managed Delta tables in Unity Catalog.

Overwrite mode is used to support repeatable full-refresh execution during project development.

In [0]:
# Persist the cleaned Silver layer as managed Delta tables
silver_orders.write.format("delta").mode("overwrite").saveAsTable(
    "workspace.default.silver_orders"
)

silver_product_dimension.write.format("delta").mode("overwrite").saveAsTable(
    "workspace.default.silver_product_dimension"
)

silver_order_items.write.format("delta").mode("overwrite").saveAsTable(
    "workspace.default.silver_order_items"
)

In [0]:
for table_name in [
    "workspace.default.silver_orders",
    "workspace.default.silver_product_dimension",
    "workspace.default.silver_order_items"
]:
    print(table_name, spark.table(table_name).count())

workspace.default.silver_orders 109467
workspace.default.silver_product_dimension 49688
workspace.default.silver_order_items 1130617


In [0]:
display(
    spark.table("workspace.default.silver_order_items")
    .limit(20)
)

order_id,user_id,product_id,product_name,aisle_id,aisle,department_id,department,add_to_cart_order,reordered,order_number,order_dow,order_hour_of_day,days_since_prior_order,eval_set
36,79431,39612,Grated Pecorino Romano Cheese,2,specialty cheeses,16,dairy eggs,1,0,23,6,18,30.0,train
36,79431,19660,Spring Water,115,water seltzer sparkling water,7,beverages,2,1,23,6,18,30.0,train
36,79431,49235,Organic Half & Half,53,cream,16,dairy eggs,3,0,23,6,18,30.0,train
36,79431,43086,Super Greens Salad,123,packaged vegetables fruits,4,produce,4,1,23,6,18,30.0,train
36,79431,46620,Cage Free Extra Large Grade AA Eggs,86,eggs,16,dairy eggs,5,1,23,6,18,30.0,train
36,79431,34497,"Prosciutto, Americano",96,lunch meat,20,deli,6,1,23,6,18,30.0,train
36,79431,48679,Organic Garnet Sweet Potato (Yam),83,fresh vegetables,4,produce,7,1,23,6,18,30.0,train
36,79431,46979,Asparagus,83,fresh vegetables,4,produce,8,1,23,6,18,30.0,train
98,56463,8859,Natural Spring Water,115,water seltzer sparkling water,7,beverages,1,1,41,3,8,14.0,train
98,56463,19731,Organic Orange Juice With Calcium & Vitamin D,31,refrigerated,7,beverages,2,1,41,3,8,14.0,train


## Silver Layer Summary

The raw Bronze datasets were transformed into three analysis-ready Silver tables.

### Tables created

- `silver_orders`
- `silver_product_dimension`
- `silver_order_items`

The Silver layer now provides clean order records, a descriptive product dimension, and a central enriched transaction table.

The next notebook aggregates these datasets into Gold business tables for customer, product, department, and ordering-behaviour analysis.